# Etapa 4 — Campaña Adaptativa Escalonada y Corridas Largas Profundas

Replica simultáneamente las dos estrategias de convergencia profunda de `pinnV3` usando **solo los modelos prometedores** seleccionados previamente:

1. **Subcampaña A (`adaptive_all_cases_campaign.py`)**:
   - Bloques progresivos `ADAM_BLOCKS = (500, 500, 500)` con comprobación del umbral **`RELAXED = 2 × STRICT`**: en cuanto se alcanza, salta anticipadamente a refinamiento **L-BFGS**.
   - Bloques `LBFGS_BLOCKS = (50, 50, 50, 50)` verificando **`STRICT`** (`error_Tmax_abs_K < 0.05`, `nrmse_pct < 0.25`, `balance_pct < 0.5`) tras cada bloque con reanudación exacta (`resume_from`).
2. **Subcampaña B (`long_v2_focused_campaign.py`)**:
   - Corrida larga profunda (`8000 Adam + 20 L-BFGS` en `full_campaign`) con `AccuracyStop(error_Tmax_abs_K=0.01, nrmse_pct=0.1, balance_pct=0.5, every=100)` sobre los 1–2 finalistas.

In [ ]:
import json
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "demo_fast"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | Workspace campaña: {CAMP_WS}")

In [ ]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        art = SolutionArtifact.load(fem_json)
        return art.predict(xy)["T"], f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
def passes_gate(metrics: dict, gate: dict) -> bool:
    return all(
        metrics.get(k) is not None and np.isfinite(metrics[k]) and float(metrics[k]) < float(lim)
        for k, lim in gate.items()
    )

promo_file = CAMP_WS / "promoted_models.json"
promo = json.loads(promo_file.read_text(encoding="utf-8-sig")) if promo_file.is_file() else {}
PROMOTED = promo.get("promoted_models", ["M1", "M1_EXACT", "CUSPR", "M0R", "M2"])
FINALISTS = promo.get("long_run_finalists", PROMOTED[:2])

if RUN_MODE == "demo_fast":
    ADAM_BLOCKS = (60, 60)
    LBFGS_BLOCKS = (10, 10)
    SEEDS_S4 = (0,)
    CASES_S4A = ("mms_constant", "cable_disk")
    MODELS_S4A = PROMOTED[:2]
    LONG_ADAM, LONG_LBFGS = 120, 15
else:
    ADAM_BLOCKS = (500, 500, 500)
    LBFGS_BLOCKS = (50, 50, 50, 50)
    SEEDS_S4 = (0, 1, 2)
    CASES_S4A = tuple(e["id"] for e in catalog.list_entries())
    MODELS_S4A = PROMOTED
    LONG_ADAM, LONG_LBFGS = 8000, 20

## Subcampaña A: Bloques Adaptativos Reanudables (`adam_i -> lbfgs_j`)

In [ ]:
adapt_dir = CAMP_WS / "stage4_adaptive_campaign"
adapt_dir.mkdir(parents=True, exist_ok=True)
adapt_summary_path = adapt_dir / "summary.json"
adapt_summary = json.loads(adapt_summary_path.read_text(encoding="utf-8-sig")) if adapt_summary_path.is_file() else {}

for case_id in CASES_S4A:
    prob = catalog.load(case_id)
    ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
    for arch in MODELS_S4A:
        for seed in SEEDS_S4:
            key = f"{case_id}:{arch}:seed{seed}"
            run_root = adapt_dir / case_id / arch / f"seed_{seed}"
            stages = []
            prev_ckpt = None
            relaxed_hit = ref_vals is None
            strict_hit = False

            for idx, b_steps in enumerate(ADAM_BLOCKS, 1):
                if ref_vals is not None and relaxed_hit:
                    break
                s_dir = run_root / f"adam_{idx}"
                cfg = TrainConfig(
                    adam_steps=b_steps,
                    lbfgs_steps=0,
                    lr=0.002,
                    width=32,
                    depth=3,
                    init_seed=seed,
                    sampling_seed=11 + seed,
                    weights={"pde": 1.0, "energy": 100.0},
                    sampling="mixed",
                    focused=True,
                    focus_surfaces=False,
                    resample_every=100,
                    resume_from=prev_ckpt,
                )
                art = PINNTrainer(cfg).train(prob, arch, s_dir, reference=ref_vals, reference_label=ref_lbl, overwrite=True)
                prev_ckpt = s_dir / "model.pt"
                m = extract_metrics(art.result)
                stages.append({"name": f"adam_{idx}", "steps": b_steps, "metrics": m, "status": art.result.get("status")})
                if ref_vals is not None and passes_gate(m, RELAXED_CRITERIA):
                    relaxed_hit = True
                    break

            for idx, l_steps in enumerate(LBFGS_BLOCKS, 1):
                s_dir = run_root / f"lbfgs_{idx}"
                cfg = TrainConfig(
                    adam_steps=0,
                    lbfgs_steps=l_steps,
                    lr=0.002,
                    width=32,
                    depth=3,
                    init_seed=seed,
                    sampling_seed=11 + seed,
                    weights={"pde": 1.0, "energy": 100.0},
                    sampling="mixed",
                    focused=True,
                    focus_surfaces=False,
                    resample_every=100,
                    resume_from=prev_ckpt,
                )
                art = PINNTrainer(cfg).train(prob, arch, s_dir, reference=ref_vals, reference_label=ref_lbl, overwrite=True)
                prev_ckpt = s_dir / "model.pt"
                m = extract_metrics(art.result)
                stages.append({"name": f"lbfgs_{idx}", "steps": l_steps, "metrics": m, "status": art.result.get("status")})
                if ref_vals is not None and passes_gate(m, STRICT_CRITERIA):
                    strict_hit = True
                    break

            adapt_summary[key] = {
                "case": case_id,
                "architecture": arch,
                "seed": seed,
                "status": "completed",
                "relaxed_reached": relaxed_hit,
                "strict_converged": strict_hit,
                "stages": stages,
            }
            write_json_atomic(adapt_summary_path, adapt_summary)
            print(f"[Etapa 4A] {key:32s} | bloques={len(stages)} | strict={strict_hit} | NRMSE={stages[-1]['metrics']['nrmse_pct']}")

## Subcampaña B: Corridas Largas Profundas con Parada Temprana (`AccuracyStop`)

In [ ]:
long_dir = CAMP_WS / "stage4_long_focused"
long_dir.mkdir(parents=True, exist_ok=True)
long_summary = {}
stop_cfg = {"error_Tmax_abs_K": 0.01, "nrmse_pct": 0.1, "balance_pct": 0.5, "every": 100, "after_phase": "adam"}

for case_id in ("mms_sine", "cable_disk"):
    prob = catalog.load(case_id)
    ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
    for arch in FINALISTS[:2]:
        key = f"{case_id}:{arch}"
        out_dir = long_dir / case_id / arch
        cfg = TrainConfig(
            adam_steps=LONG_ADAM,
            lbfgs_steps=LONG_LBFGS,
            lr=0.002,
            width=32,
            depth=3,
            weights={"pde": 1.0, "energy": 100.0},
            sampling="mixed",
            focused=True,
            focus_surfaces=False,
            resample_every=100,
            accuracy_stop=stop_cfg if ref_vals is not None else None,
        )
        art = PINNTrainer(cfg).train(prob, arch, out_dir, reference=ref_vals, reference_label=ref_lbl, overwrite=True)
        long_summary[key] = {
            "case": case_id,
            "architecture": arch,
            "status": art.result.get("status"),
            "accuracy_stop": art.result.get("accuracy_stop"),
            **extract_metrics(art.result),
        }
        print(f"[Etapa 4B] {key:24s} | NRMSE={long_summary[key]['nrmse_pct']} | dTmax={long_summary[key]['error_Tmax_abs_K']} K")

write_json_atomic(long_dir / "summary.json", long_summary)